## IMPORT LIBRARIES

In [26]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## LOAD DATASET

In [27]:
movies=pd.read_csv('tmdb_5000_movies.csv')
credits=pd.read_csv('tmdb_5000_credits.csv')

In [28]:
movies.head()
credits.head()

,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,49026,The Dark Knight Rises,"[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,49529,John Carter,"[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


## MERGE DATASETS

In [29]:
movies = movies.merge(credits, on='title')

## SELECT IMPORTANT COLUMNS

In [ ]:
movies = movies[['movie_id',
                 'title',
                 'overview',
                 'genres',
                 'keywords',
                 'cast',
                 'crew',
                 'vote_average']]

## CHECK MISSING VALUES

In [31]:
movies.isnull().sum()

movie_id    0
title       0
overview    3
genres      0
keywords    0
cast        0
crew        0
dtype: int64

In [32]:
movies.dropna(inplace=True)

In [33]:
movies.isnull().sum()

movie_id    0
title       0
overview    0
genres      0
keywords    0
cast        0
crew        0
dtype: int64

### CONVERT COLUMNS

Columns like:

genres
keywords
cast
crew

contain dictionaries/lists as strings.

Need to extract names.

In [34]:
import ast

def convert(text):

    L = []

    for i in ast.literal_eval(text):

        L.append(i['name'])

    return L

In [35]:
movies['genres'] = movies['genres'].apply(convert)

movies['keywords'] = movies['keywords'].apply(convert)

movies['cast'] = movies['cast'].apply(convert)

### EXTRACT DIRECTOR

In [36]:
def fetch_director(text):

    L = []

    for i in ast.literal_eval(text):

        if i['job'] == 'Director':

            L.append(i['name'])

    return L

In [37]:
movies['crew'] = movies['crew'].apply(fetch_director)

In [38]:
movies['overview'] = movies['overview'].apply(lambda x:x.split())

## REMOVE SPACES

In [39]:
def collapse(L):

    L1 = []

    for i in L:

        L1.append(i.replace(" ",""))

    return L1

In [40]:
movies['cast'] = movies['cast'].apply(collapse)

movies['crew'] = movies['crew'].apply(collapse)

movies['genres'] = movies['genres'].apply(collapse)

movies['keywords'] = movies['keywords'].apply(collapse)

### CREATE TAGS COLUMN

In [41]:
movies['tags'] = movies['overview'] + \
                 movies['genres'] + \
                 movies['keywords'] + \
                 movies['cast'] + \
                 movies['crew']

In [42]:
new_df = movies[['movie_id',
                 'title',
                 'tags',
                 'vote_average']]

KeyError: "['vote_average'] not in index"

In [ ]:
new_df['tags'] = new_df['tags'].apply(lambda x:" ".join(x))

C:\Users\Sarika\AppData\Local\Temp\ipykernel_29664\3089450492.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x:" ".join(x))


In [ ]:
new_df['tags'] = new_df['tags'].apply(lambda x:x.lower())

C:\Users\Sarika\AppData\Local\Temp\ipykernel_29664\3214958533.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x:x.lower())


### TEXT VECTORIZATION

In [ ]:
cv = CountVectorizer(max_features=5000,
                     stop_words='english')

In [ ]:
vectors = cv.fit_transform(new_df['tags']).toarray()

### CALCULATE COSINE SIMILARITY

In [ ]:
similarity = cosine_similarity(vectors)

### RECOMMENDATION FUNCTION

In [ ]:
def recommend(movie):

    movie_index = new_df[new_df['title'] == movie].index[0]

    distances = similarity[movie_index]

    movies_list = sorted(list(enumerate(distances)),
                         reverse=True,
                         key=lambda x:x[1])[1:6]

    for i in movies_list:

        print(new_df.iloc[i[0]].title)

In [ ]:
recommend('Avatar')

Lifeforce
Aliens vs Predator: Requiem
Battle: Los Angeles
Titan A.E.
Independence Day


In [ ]:
import pickle

pickle.dump(new_df, open('movies.pkl', 'wb'))

pickle.dump(similarity, open('similarity.pkl', 'wb'))

In [43]:
movies.columns

Index(['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew',
       'tags'],
      dtype='object')